# LongestPassword

[Codility task](https://app.codility.com/programmers/trainings/1/longest_password/) - Exercise 1, 2015 Contest

Split `S` at the spaces into words. A word is a valid password when it contains only `a-z`, `A-Z`, `0-9`, an **even** number of letters and an **odd** number of digits. `solution(S)` returns the length of the longest valid word, or `-1`.

| S | result |
|---|--------|
| `"test 5 a0A pass007 ?xy1"` | 7 (`"5"`, `"a0A"` and `"pass007"` are valid) |

Assumptions: `N` in `[1..200]`, printable ASCII characters and spaces.

## Idea

1. `S.split(" ")` gives exactly `K + 1` words for `K` spaces (empty words appear with consecutive spaces; they are never valid because they have zero digits, which is even).
2. Check each word: only ASCII alphanumerics, even number of letters, odd number of digits.
3. Keep the longest valid length, or `-1`.

An explicit ASCII check is used instead of `str.isalnum()`, which also accepts non-ASCII letters.

Time O(N), space O(N).

In [ ]:
import string

LETTERS = set(string.ascii_letters)
DIGITS = set(string.digits)


def is_valid(word):
    if not word:
        return False
    letters = digits = 0
    for ch in word:
        if ch in LETTERS:
            letters += 1
        elif ch in DIGITS:
            digits += 1
        else:
            return False
    return letters % 2 == 0 and digits % 2 == 1


def solution(S):
    best = -1
    for word in S.split(" "):
        if is_valid(word):
            best = max(best, len(word))
    return best

## Official example

In [ ]:
assert solution("test 5 a0A pass007 ?xy1") == 7
print("example OK")

## Edge cases

In [ ]:
assert solution("a") == -1                      # one letter (odd) and no digit (even)
assert solution("1") == 1                       # zero letters (even), one digit (odd)
assert solution("a1") == -1                     # one letter: odd
assert solution("ab1") == 3
assert solution("ab12") == -1                   # two digits: even
assert solution("123") == 3                     # zero letters, three digits
assert solution(" ") == -1                      # two empty words
assert solution("  ab1  ") == 3                 # leading, trailing and repeated spaces
assert solution("a!1") == -1                    # forbidden character
assert solution("ab1_") == -1
assert solution("AZaz9") == 5                   # upper and lower case letters count as letters
assert solution("x ab1 y 12345") == 5           # the longest valid word wins
assert solution("e\u00e91") == -1                # non-ASCII letter (isalnum() would wrongly accept it)
print("edge cases OK")

## Reference with regular expressions

Another way to write the same rules: `fullmatch` for the alphabet and `findall` to count letters and digits. Compared on 6000 random strings made of a few letters, digits, spaces and forbidden symbols.

In [ ]:
import random
import re


def brute(S):
    best = -1
    for word in S.split(" "):
        if (re.fullmatch("[A-Za-z0-9]+", word)
                and len(re.findall("[A-Za-z]", word)) % 2 == 0
                and len(re.findall("[0-9]", word)) % 2 == 1):
            best = max(best, len(word))
    return best


rng = random.Random(1)
alphabet = "aZ09  ?_-"
for _ in range(6000):
    s = "".join(rng.choice(alphabet) for _ in range(rng.randint(1, 20)))
    assert solution(s) == brute(s), repr(s)
print("matches the regular-expression reference on 6000 random strings")

## Performance (the statement limits `N` to 200; here a much larger input)

In [ ]:
import time

S = " ".join(["pass007"] * 100_000 + ["x"])
start = time.perf_counter()
assert solution(S) == 7
print(f"{len(S):,} characters in {time.perf_counter() - start:.3f}s")

## Notes

- "Valid password" is three independent conditions; counting letters and digits in one pass keeps it simple.
- `split(" ")` (with an explicit space) preserves empty words; `split()` would drop them, which is harmless here but changes the "K + 1 words" model.
- Prefer explicit ASCII sets to `isalnum()` / `isalpha()` when the specification talks about specific ranges.